In [ ]:
!pip install -q gymnasium

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

print("Libraries imported successfully!")

In [ ]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode=None
)

print("Environment: FrozenLake-v1")
print("Number of states:", env.observation_space.n)
print("Number of actions:", env.action_space.n)

print("\nAction meanings:")
print("0 = Left")
print("1 = Down")
print("2 = Right")
print("3 = Up")

In [ ]:
env_map = [
    ['S', 'F', 'F', 'F'],
    ['F', 'H', 'F', 'H'],
    ['F', 'F', 'F', 'H'],
    ['H', 'F', 'F', 'G']
]

print("FrozenLake 4x4 Map:\n")

for row in env_map:
    print(" | ".join(row))

print("\nS = Start")
print("F = Frozen / Safe")
print("H = Hole")
print("G = Goal")

In [ ]:
gamma = 0.99
epsilon = 0.1

num_episodes = 20000

# Number of states and actions
n_states = env.observation_space.n
n_actions = env.action_space.n

# Q-table
Q = np.zeros((n_states, n_actions))

# Store returns for each state-action pair
returns_sum = defaultdict(float)
returns_count = defaultdict(float)

print("Gamma:", gamma)
print("Epsilon:", epsilon)
print("Episodes:", num_episodes)
print("Q-table shape:", Q.shape)

In [ ]:
def epsilon_greedy_action(state, Q, epsilon):
    """
    Select an action using epsilon-greedy policy.

    With probability epsilon:
        Explore by selecting a random action.

    Otherwise:
        Exploit by selecting the action with highest Q-value.

    This function implements the epsilon-greedy policy for action selection. This policy is crucial for balancing exploration (trying new actions to discover better rewards) and exploitation (choosing actions that are already known to yield high rewards).

    -   **Exploration**: With a probability of `epsilon`, the agent chooses a random action from the available actions. This helps the agent discover new paths or actions that might lead to higher rewards.
    -   **Exploitation**: With a probability of `1 - epsilon`, the agent chooses the action that has the highest Q-value for the current state. The Q-value represents the expected cumulative reward for taking a particular action in a given state and then following the optimal policy thereafter. If multiple actions have the same maximum Q-value, one is chosen randomly among them to avoid bias.
    """

    if np.random.random() < epsilon:
        return np.random.randint(Q.shape[1])

    max_value = np.max(Q[state])

    # Select randomly among equally good actions
    best_actions = np.flatnonzero(Q[state] == max_value)

    return np.random.choice(best_actions)

In [ ]:
def generate_episode(env, Q, epsilon):
    """
    Generate one complete episode using epsilon-greedy policy.

    Returns:
        List of (state, action, reward)

    This function is responsible for simulating one complete episode of the agent interacting with the environment. An 'episode' starts from an initial state and continues until a terminal state is reached (e.g., the agent falls into a hole or reaches the goal).

    -   **Initialization**: The episode starts by resetting the environment, which places the agent in the initial state.
    -   **Action Selection**: At each step, the `epsilon_greedy_action` function is used to select an action, balancing exploration and exploitation.
    -   **Environment Interaction**: The chosen action is then executed in the environment using `env.step()`, which returns the next state, the reward received, and whether the episode has terminated or been truncated.
    -   **Episode Recording**: Each (state, action, reward) tuple is recorded to build a complete history of the episode. This history is later used by the Monte Carlo control algorithm to update Q-values.
    """

    episode = []

    state, info = env.reset()

    done = False

    while not done:

        action = epsilon_greedy_action(
            state,
            Q,
            epsilon
        )

        next_state, reward, terminated, truncated, info = env.step(action)

        episode.append((state, action, reward))

        state = next_state

        done = terminated or truncated

    return episode

In [ ]:
def monte_carlo_control(
    env,
    num_episodes,
    gamma=0.99,
    epsilon=0.1
):
    """
    This is the core function that implements the First-Visit Monte Carlo control algorithm to learn the optimal Q-values and, consequently, the optimal policy for the FrozenLake environment.

    #### Key Steps:

    1.  **Initialization**:
        -   `Q` (Q-table): A NumPy array is initialized with zeros, where `Q[state, action]` will store the estimated expected return for taking `action` in `state`.
        -   `returns_sum` and `returns_count`: Dictionaries are used to store the sum of returns and the count of visits for each (state, action) pair. These are necessary for calculating the average return.
        -   `episode_rewards` and `episode_lengths`: Lists to track the total reward and length of each episode, useful for plotting learning progress.

    2.  **Episode Generation Loop**: The algorithm runs for a specified `num_episodes`.
        -   In each iteration, `generate_episode()` is called to simulate an entire episode using the current Q-table and epsilon-greedy policy.
        -   The total reward for the episode is calculated and stored.

    3.  **Q-value Update (Backward Pass)**:
        -   The algorithm iterates through the episode in reverse order (from the last step to the first).
        -   **Return Calculation (`G`)**: The discounted cumulative return (`G`) is calculated for each step. This is the sum of discounted rewards from that step until the end of the episode. The `gamma` parameter determines the discount factor, giving more weight to immediate rewards.
        -   **First-Visit Condition**: To implement *First-Visit Monte Carlo*, Q-values are only updated for the first occurrence of a (state, action) pair within an episode. This avoids issues with multiple visits to the same state-action pair within a single episode.
        -   **Q-value Averaging**: When a (state, action) pair is visited for the first time in an episode, its `G` is added to `returns_sum` and its `returns_count` is incremented. The Q-value for that (state, action) pair is then updated as the average of all recorded returns for that pair.
    """

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    # Initialize Q-values
    Q = np.zeros((n_states, n_actions))

    # Store cumulative returns
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)

    episode_rewards = []
    episode_lengths = []

    for episode_number in range(num_episodes):

        # Generate episode
        episode = generate_episode(
            env,
            Q,
            epsilon
        )

        # Track total reward
        total_reward = sum(
            reward for _, _, reward in episode
        )

        episode_rewards.append(total_reward)
        episode_lengths.append(len(episode))

        # Calculate returns backwards
        G = 0

        # Used for First-Visit MC
        visited_state_actions = set()

        for t in reversed(range(len(episode))):

            state, action, reward = episode[t]

            # Return
            G = gamma * G + reward

            # First-Visit condition
            if (state, action) not in visited_state_actions:

                visited_state_actions.add(
                    (state, action)
                )

                returns_sum[(state, action)] += G

                returns_count[(state, action)] += 1

                # Update Q-value
                Q[state, action] = (
                    returns_sum[(state, action)]
                    /
                    returns_count[(state, action)]
                )

    return Q, episode_rewards, episode_lengths

In [ ]:
# Training Execution
# This cell executes the `monte_carlo_control` function, initiating the learning process for the FrozenLake environment.
# The function returns:
# -   `Q`: The learned Q-table, containing the estimated optimal Q-values for each state-action pair.
# -   `episode_rewards`: A list of the total rewards obtained in each episode during training.
# -   `episode_lengths`: A list of the lengths (number of steps) of each episode during training.
# The 'Training completed!' message indicates that the Monte Carlo control algorithm has finished running for the specified number of episodes.
Q, episode_rewards, episode_lengths = monte_carlo_control(
    env=env,
    num_episodes=num_episodes,
    gamma=gamma,
    epsilon=epsilon
)

print("Training completed!")

In [ ]:
np.set_printoptions(
    precision=4,
    suppress=True
)

print("Learned Q-Table:\n")
print(Q)
# Learned Q-Table:
# This output displays the final Q-table learned by the Monte Carlo control algorithm after `num_episodes` of training.
# Each row in the Q-table corresponds to a state in the FrozenLake environment (0 to 15), and each column corresponds to an action (0: Left, 1: Down, 2: Right, 3: Up).
# The values in the table represent the estimated expected future reward (Q-value) for taking a particular action in a given state.
#
# -   **Interpretation**: A higher Q-value for a specific (state, action) pair suggests that taking that action from that state is expected to lead to greater cumulative rewards in the long run.
# -   **`np.set_printoptions`**: This function is used to format the NumPy array output for better readability, suppressing small values and setting the precision to 4 decimal places.

In [ ]:
policy = np.argmax(Q, axis=1)

print("Learned Policy:")
print(policy)

print("\nAction Mapping:")
print("0 = Left")
print("1 = Down")
print("2 = Right")
print("3 = Up")
# Derived Policy from Q-Table
# This cell demonstrates how to derive the optimal policy from the learned Q-table. The policy dictates the best action to take in each state.
#
# -   **`np.argmax(Q, axis=1)`**: For each state (row) in the Q-table, `np.argmax` finds the index of the action (column) that has the maximum Q-value. This action is considered the optimal action for that state according to the learned Q-values.
# -   **Policy Output**: The resulting `policy` array contains the optimal action (0, 1, 2, or 3) for each of the 16 states. This represents the agent's learned strategy to navigate the FrozenLake environment.

In [ ]:
action_arrows = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

print("Learned Policy on FrozenLake:\n")

for row in range(4):

    row_policy = []

    for col in range(4):

        state = row * 4 + col

        cell = env_map[row][col]

        if cell == 'H':
            row_policy.append(' H ')
        elif cell == 'G':
            row_policy.append(' G ')
        else:
            arrow = action_arrows[policy[state]]
            row_policy.append(f' {arrow} ')

    print("|".join(row_policy))
# Visualizing the Learned Policy on FrozenLake
# This cell provides a textual visualization of the learned policy directly on the FrozenLake 4x4 grid.
#
# -   **`action_arrows`**: A dictionary maps the numerical actions (0, 1, 2, 3) to their corresponding arrow symbols (←, ↓, →, ↑) for intuitive representation.
# -   **Grid Iteration**: The code iterates through each cell of the 4x4 grid.
#     -   If a cell is a 'Hole' (`H`) or the 'Goal' (`G`), it's marked as such.
#     -   For 'Start' (`S`) and 'Frozen' (`F`) cells, the learned optimal action (represented by an arrow) for that state is displayed.
# This visualization helps to quickly understand the agent's preferred movement strategy across the environment.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.set_xlim(0, 4)
ax.set_ylim(0, 4)

ax.set_xticks(np.arange(5))
ax.set_yticks(np.arange(5))
ax.grid(True)

ax.set_title(
    "FrozenLake-v1 - Learned Monte Carlo Policy",
    fontsize=14
)

for row in range(4):

    for col in range(4):

        state = row * 4 + col

        x = col + 0.5
        y = 3 - row + 0.5

        cell = env_map[row][col]

        if cell == 'H':
            text = "H"
        elif cell == 'G':
            text = "G"
        elif cell == 'S':
            text = "S\n" + action_arrows[policy[state]]
        else:
            text = action_arrows[policy[state]]

        ax.text(
            x,
            y,
            text,
            ha='center',
            va='center',
            fontsize=22
        )

ax.set_xlabel("Column")
ax.set_ylabel("Row")

plt.show()
# Graphical Visualization of the Learned Policy
# This cell generates a more visually appealing representation of the learned policy on the FrozenLake environment using `matplotlib`.
#
# -   **Grid Setup**: A 4x4 grid is created using `plt.subplots`, with appropriate limits, ticks, and grid lines.
# -   **Policy Overlay**: For each cell in the grid:
#     -   If the cell is a 'Hole' (`H`) or the 'Goal' (`G`), it's labeled with 'H' or 'G' respectively.
#     -   If the cell is the 'Start' (`S`), it's labeled with 'S' and the optimal action arrow.
#     -   For 'Frozen' (`F`) cells, only the optimal action arrow is displayed.
# -   **Text Annotation**: The `ax.text()` function is used to place the labels and arrows in the center of each grid cell with a large font size for clarity.
# -   **Labels and Title**: The plot is given a title and axis labels for better context.
# This graphical representation makes it easier to grasp the agent's learned navigation strategy and observe the flow of the policy.

In [ ]:
window = 500

moving_average = np.convolve(
    episode_rewards,
    np.ones(window) / window,
    mode='valid'
)

plt.figure(figsize=(10, 5))

plt.plot(
    moving_average
)

plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.title(
    "Monte Carlo Learning Progress"
)

plt.grid(True)

plt.show()
# Monte Carlo Learning Progress (Moving Average of Rewards)
# This plot visualizes the learning progress of the Monte Carlo control algorithm over `num_episodes`. Instead of plotting the reward for each individual episode (which can be very noisy), a moving average is used to smooth out the curve and show the general trend.
#
# -   **`window = 500`**: A window size of 500 means that each point on the `moving_average` curve represents the average reward over the preceding 500 episodes.
# -   **`np.convolve()`**: This function is used to compute the moving average. It performs a convolution of the `episode_rewards` array with a uniform window (an array of ones divided by the window size).
# -   **Plot Interpretation**: The plot shows how the average reward obtained per episode changes as the agent gains more experience. An upward trend indicates that the agent is learning to achieve higher rewards, suggesting that the Q-table is being refined towards an optimal policy. The eventual flattening of the curve indicates convergence or stabilization of the learning process.

In [ ]:
def evaluate_policy(env, Q, num_episodes=1000):
    """
    This function is designed to evaluate the performance of the learned policy (derived from the final Q-table) by running a specified number of test episodes. Unlike the training phase which uses an epsilon-greedy policy, evaluation uses a *greedy* policy to assess the agent's performance given its learned knowledge.

    #### Key Steps:

    1.  **Initialization**: Lists `rewards` and a counter `successes` are initialized to track the total reward and the number of successful episodes.
    2.  **Evaluation Episodes Loop**: The function runs for `num_episodes` without further exploration.
        -   **Environment Reset**: Each evaluation episode starts by resetting the environment.
        -   **Greedy Action Selection**: At each step, the agent always chooses the action with the highest Q-value for the current state (`np.argmax(Q[state])`). There is no random exploration (epsilon is effectively 0).
        -   **Environment Interaction**: The environment is stepped through with the greedy action, accumulating `total_reward`.
        -   **Success Check**: An episode is considered a success if `total_reward > 0`. In FrozenLake, this typically means reaching the goal state (which gives a reward of 1) without falling into a hole.
    3.  **Performance Metrics**: After all evaluation episodes, the `average_reward` (mean of all episode rewards) and `success_rate` (percentage of successful episodes) are calculated and returned.
    """

    rewards = []
    successes = 0

    for _ in range(num_episodes):

        state, info = env.reset()

        total_reward = 0
        done = False

        while not done:

            # Greedy action
            action = np.argmax(Q[state])

            next_state, reward, terminated, truncated, info = env.step(action)

            total_reward += reward

            state = next_state

            done = terminated or truncated

        rewards.append(total_reward)

        if total_reward > 0:
            successes += 1

    average_reward = np.mean(rewards)
    success_rate = successes / num_episodes

    return average_reward, success_rate

In [ ]:
average_reward, success_rate = evaluate_policy(
    env,
    Q,
    num_episodes=1000
)

print("Evaluation Results")
print("------------------")
print("Number of evaluation episodes:", 1000)
print("Average Reward:", average_reward)
print("Success Rate:", success_rate * 100, "%")
# Policy Evaluation Results
# This cell executes the `evaluate_policy` function and prints the performance metrics of the learned policy.
#
# -   **`average_reward`**: The average reward achieved by the agent over 1000 evaluation episodes. A higher value indicates a more effective policy.
# -   **`success_rate`**: The percentage of evaluation episodes where the agent successfully reached the goal (i.e., received a positive reward). This is a direct measure of the policy's reliability in completing the task.
# These metrics provide a quantitative assessment of how well the Monte Carlo control algorithm has learned to navigate the FrozenLake environment.

In [ ]:
def run_sample_episode(env, Q):
    """
    This function simulates a single episode using the *learned greedy policy* from the final Q-table and records the path taken by the agent and the total reward obtained. This is useful for observing the agent's behavior directly.

    #### Key Steps:

    1.  **Environment Reset**: The episode starts by resetting the environment.
    2.  **Path Tracking**: The initial state is added to the `path` list.
    3.  **Greedy Action Loop**: The agent iteratively takes actions:
        -   It selects the greedy action (highest Q-value) for the current state.
        -   It interacts with the environment (`env.step()`) to get the next state, reward, and done status.
        -   The `next_state` is added to the `path`.
        -   The `reward` is accumulated into `total_reward`.
        -   The process continues until a terminal state is reached.

    #### Sample Episode Output:

    -   **`Path`**: A list of states the agent visited from start to finish. This can show the trajectory the agent takes.
    -   **`Total Reward`**: The sum of rewards received during this specific sample episode. For FrozenLake, this is typically 1 for reaching the goal or 0 for falling into a hole/timeout.
    """

    state, info = env.reset()

    path = [state]

    done = False

    total_reward = 0

    while not done:

        action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, info = env.step(action)

        path.append(next_state)

        total_reward += reward

        state = next_state

        done = terminated or truncated

    return path, total_reward


path, reward = run_sample_episode(env, Q)

print("Sample Episode")
print("----------------")

print("Path:", path)
print("Total Reward:", reward)

In [ ]:
print("=" * 50)
print("EXPERIMENT 6 - FINAL RESULTS")
print("=" * 50)

print("\nEnvironment:")
print("FrozenLake-v1")

print("\nParameters:")
print("Gamma =", gamma)
print("Epsilon =", epsilon)
print("Episodes =", num_episodes)

print("\nQ-Table:")
print(Q)

print("\nPolicy:")
print(policy)

print("\nEvaluation:")
print("Average Reward =", average_reward)
print("Success Rate =", success_rate * 100, "%")

print("\nLearned Policy:")
for row in range(4):

    row_policy = []

    for col in range(4):

        state = row * 4 + col
        cell = env_map[row][col]

        if cell == 'H':
            row_policy.append('H')
        elif cell == 'G':
            row_policy.append('G')
        else:
            row_policy.append(action_arrows[policy[state]])

    print(" | ".join(row_policy))

print("=" * 50)
# Experiment 6 - Final Results Summary
# This cell provides a comprehensive summary of the entire Monte Carlo control experiment, bringing together all the key information and results in one place.
#
# #### Summary Includes:
#
# -   **Environment**: Details about the FrozenLake-v1 environment.
# -   **Parameters**: The values used for `gamma` (discount factor), `epsilon` (exploration rate), and `num_episodes` (number of training episodes).
# -   **Q-Table**: The final learned Q-table, which is the core output of the Monte Carlo control algorithm.
# -   **Policy**: The optimal policy derived from the Q-table, showing the best action for each state.
# -   **Evaluation**: The quantitative performance metrics of the learned policy, including the `Average Reward` and `Success Rate`.
# -   **Learned Policy (Grid View)**: A visual representation of the policy directly on the FrozenLake grid, using arrows to indicate the optimal action for each state (excluding holes and the goal).
#
# This summary acts as a final report for the experiment, allowing for a quick review of the setup, learned behavior, and overall performance of the reinforcement learning agent.

### Experiment 6 - Final Results Summary

This cell provides a comprehensive summary of the entire Monte Carlo control experiment, bringing together all the key information and results in one place.

#### Summary Includes:

-   **Environment**: Details about the FrozenLake-v1 environment.
-   **Parameters**: The values used for `gamma` (discount factor), `epsilon` (exploration rate), and `num_episodes` (number of training episodes).
-   **Q-Table**: The final learned Q-table, which is the core output of the Monte Carlo control algorithm.
-   **Policy**: The optimal policy derived from the Q-table, showing the best action for each state.
-   **Evaluation**: The quantitative performance metrics of the learned policy, including the `Average Reward` and `Success Rate`.
-   **Learned Policy (Grid View)**: A visual representation of the policy directly on the FrozenLake grid, using arrows to indicate the optimal action for each state (excluding holes and the goal).

This summary acts as a final report for the experiment, allowing for a quick review of the setup, learned behavior, and overall performance of the reinforcement learning agent.